<h1 style="text-align:center">🔬 TD noté — Identification de modèles de comportement</h1>
<p style="text-align:center"><b>Plasticité · Écrouissage · Corrélation calcul / essais</b><br>
<i>Simulations éléments finis à un seul élément : essais de traction et chargements cycliques</i></p>

---

| Exercice | Thème | Données |
|:--:|:--|:--|
| **2** | Introduction à l'identification de paramètres | essais simulés — traction et cyclique |
| **3** | Corrélation calcul/essais en plasticité | fonte — traction et cyclique |

<div class="alert alert-block alert-warning">

**⚠️ Prérequis — à vérifier avant d'exécuter le notebook**

1. Le logiciel est installé en suivant le **tutoriel d'installation** fourni.
2. Le notebook est lancé **depuis la racine du projet**, qui doit contenir :
   - `src/code/` — modules Python (`functions.py`, `models.py`)
   - `data/` — données expérimentales (`.csv`)
   - `sim_results/` — dossier de sortie des simulations (`.txt`, `.png`)
3. Les cellules sont exécutées **dans l'ordre**, en commençant par la cellule `autoreload` ci-dessous.

</div>

<div class="alert alert-block alert-info">

**ℹ️ Mode d'emploi**

Chaque question de simulation suit le même schéma en **3 cellules** :

| Cellule | Rôle |
|:--:|:--|
| 1️⃣ `model_dict` → `run_simulation()` | définit les paramètres matériau et le chargement, puis lance le calcul |
| 2️⃣ `graph_e_and_sigma_time` | trace $\varepsilon(t)$ et $\sigma(t)$ |
| 3️⃣ `graph_e_and_sigma` | superpose la courbe $\sigma$–$\varepsilon$ simulée aux données et enregistre la figure dans `sim_results/` |

👉 Pour identifier un modèle, **modifiez uniquement les valeurs du bloc `"material_property"`**, puis ré-exécutez les 3 cellules.
Les valeurs déjà présentes sont de simples **valeurs de départ**, pas une solution.

**🎨 Code couleur :** 🔵 consigne / information · 🟡 attention / indice · 🟢 votre réponse

**✍️ Pour répondre :** **double-cliquez** sur un cadre vert *VOTRE RÉPONSE*, remplacez les textes *à compléter* par vos valeurs et votre commentaire, puis validez avec **Maj + Entrée**.

**🧭 Navigation :** ouvrez la **table des matières** dans la barre latérale gauche de JupyterLab, et cliquez sur la flèche à gauche d'un titre pour **replier une section** (par ex. l'exercice 3 pendant que vous travaillez sur le 2). Les sorties texte des simulations sont repliées : cliquez dessus pour les afficher.

</div>

<details>
<summary><b>📘 Aide-mémoire : paramètres et modèles</b> (cliquer pour déplier)</summary>

| Clé dans `material_property` | Symbole | Signification |
|:--|:--:|:--|
| `YoungModulus` | $E$ | module d'Young (MPa) |
| `PoissonRatio` | $\nu$ | coefficient de Poisson |
| `R0` | $R_0$ | limite d'élasticité initiale (MPa) |
| `C` | $C$ | module d'écrouissage cinématique linéaire |
| `K` | $K$ | constante de durcissement isotrope linéaire |
| `C[0]`, `C[1]` | $C_1,\ C_2$ | modules d'écrouissage cinématique non linéaire |
| `D_alpha[0]`, `D_alpha[1]` | $D_1,\ D_2$ | termes de rappel associés |
| `Q`, `b` | $Q,\ b$ | amplitude et vitesse de saturation de l'écrouissage isotrope non linéaire |

| `behavior_name` | Modèle |
|:--|:--|
| `asterlinearkinematic` | écrouissage cinématique linéaire |
| `asterlinearisotropic` | écrouissage isotrope linéaire |
| `asternonlinearisotropicnonlinearkinematic` | isotrope non linéaire + cinématique non linéaire — un terme est désactivé en mettant ses paramètres à `0` |

</details>

In [ ]:
%load_ext autoreload
%autoreload 2

<h2 style="background-color:#1f4e79;color:#fff;padding:10px;border-radius:6px;text-align:center">🧪 Exercice 2 — Introduction à l'identification de paramètres de modèle de comportement</h2>

Cet exercice constitue la **première partie du TD noté** (sous forme de DM). Il s'appuie sur le tutoriel d'installation et d'utilisation du logiciel qui vous a été fourni : celui-ci permet de simuler, à partir d'un calcul éléments finis à un seul élément, des essais de **traction**, de **traction-compression** et de **relaxation**.

Un même modèle (les mêmes équations de comportement) peut mener à des réponses mécaniques très différentes selon l'identification de ses paramètres. L'ensemble des données expérimentales et simulées est visualisé avec Matplotlib.

**🎯 Objectifs**
- vous familiariser avec l'utilisation du logiciel ;
- comprendre l'effet de chacun des paramètres du modèle ;
- **retrouver le modèle et les paramètres** utilisés pour générer les données.

<div class="alert alert-block alert-info">

**ℹ️ À savoir :** toutes les données de cet exercice ont été **générées numériquement** avec un modèle donné. Elles sont donc sans bruit : une identification exacte est possible.

</div>

<h3 style="border-left:6px solid #1f4e79;padding-left:10px">📈 2.1 — Autour de l'essai de traction</h3>

In [ ]:
from src.code.functions import draw_data
draw_data(input_data = "data/exercice1_traction.csv", figure_title = 'FIGURE 2: Chargement et réponse mécanique étudiés')

<div class="alert alert-block alert-info">

**📌 Chargement :** essai de traction **piloté en déformation, de 0 à 0,02 mm/mm (soit 2 %) en 200 s**.

</div>

### ❓ Questions

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.1 Traction · Q1 / Q6</span>

#### Q1 · Module d'Young

<div class="alert alert-block alert-info">

Estimez tout d'abord la valeur du **module d'Young** $E$ en considérant les premiers points expérimentaux, dans le régime linéaire.

</div>

> 💡 Les essais étant simulés, il n'y a **pas de bruit** : vous devez trouver une valeur **exacte**.

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.1 · Q1</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

**Module d'Young :** $E =$ *à compléter* MPa

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.1 Traction · Q2 / Q6</span>

#### Q2 · Écrouissage cinématique linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **cinématique linéaire** : faites varier la limite d'élasticité et le module d'écrouissage. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ?

🧩 **Modèle :** `asterlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `C`

</div>

In [ ]:
#########################################################################
### Simulation d'un essai de traction - écrouissage cinématique  linéaire 
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 100000,
    "PoissonRatio" : 0.3,
    "R0" : 500,
    "C" : 8000
  },
  "output" : {
    "filename" : "sim_results/traction_ex2.1_q2.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 200,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        2 : {
            "time" : 200,
            "load" : 0.02,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

lk_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asterlinearkinematic')
lk_instance.print_input()
lk_instance.run_simulation()
data_lk = lk_instance.print_output()

In [ ]:
lk_instance.graph_e_and_sigma_time(data_lk)

In [ ]:
lk_instance.graph_e_and_sigma(dataframe = data_lk, 
                              input_data = "data/exercice1_traction.csv", 
                              savefig_name = "sim_results/traction_ex2.1_q2.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.1 · Q2</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `C` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.1 Traction · Q3 / Q6</span>

#### Q3 · Écrouissage cinématique non linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **cinématique non linéaire** : faites varier la limite d'élasticité, le module d'écrouissage et le terme de rappel. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ?

🧩 **Modèle :** `asternonlinearisotropicnonlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `C[0]`, `D_alpha[0]` (garder `Q = b = 0`)

</div>

In [ ]:
#########################################################################
### Simulation d'un essai de traction - écrouissage cinématique non linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 20000,
    "PoissonRatio" : 0.3,
    "R0" : 10000,
    "C[0]" : 1000,
    "D_alpha[0]" : 1000.,
    "Q" : 0,
    "b" : 0,
    "C[1]" : 0,
    "D_alpha[1]" : 0
  },
  "output" : {
    "filename" : "sim_results/traction_ex2.1_q3.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 200,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        2 : {
            "time" : 200,
            "load" : 0.02,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

nlk_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
nlk_instance.print_input()
nlk_instance.run_simulation()
data_nlk = nlk_instance.print_output()

In [ ]:
nlk_instance.graph_e_and_sigma_time(data_nlk)

In [ ]:
nlk_instance.graph_e_and_sigma(dataframe = data_nlk, 
                               input_data = "data/exercice1_traction.csv", 
                               savefig_name = "sim_results/traction_ex2.1_q3.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.1 · Q3</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `C[0]` | *à compléter* |
| `D_alpha[0]` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.1 Traction · Q4 / Q6</span>

#### Q4 · Écrouissage isotrope linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **isotrope linéaire** : faites varier la limite d'élasticité et la constante de durcissement. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ? Quelles différences obtenez-vous avec un modèle cinématique ?

🧩 **Modèle :** `asterlinearisotropic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `K`

</div>

In [ ]:
#########################################################################
### Simulation d'un essai de traction - écrouissage isotrope linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 20000,
    "PoissonRatio" : 0.3,
    "R0" : 10000,
    "K" : 1000,
  },
  "output" : {
    "filename" : "sim_results/traction_ex2.1_q4.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 200,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        2 : {
            "time" : 200,
            "load" : 0.02,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

li_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asterlinearisotropic')
li_instance.print_input()
li_instance.run_simulation()
data_li = li_instance.print_output()

In [ ]:
li_instance.graph_e_and_sigma_time(data_li)

In [ ]:
li_instance.graph_e_and_sigma(dataframe = data_li, 
                              input_data = "data/exercice1_traction.csv", 
                              savefig_name = "sim_results/traction_ex2.1_q4.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.1 · Q4</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `K` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.1 Traction · Q5 / Q6</span>

#### Q5 · Écrouissage isotrope non linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **isotrope non linéaire** : faites varier la limite d'élasticité, la constante de durcissement et le terme de rappel. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ?

🧩 **Modèle :** `asternonlinearisotropicnonlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `Q`, `b` (garder `C[i] = D_alpha[i] = 0`)

</div>

In [ ]:
#########################################################################
### Simulation d'un essai de traction - écrouissage isotrope non linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 20000,
    "PoissonRatio" : 0.3,
    "R0" : 10000,
    "Q" : 1000,
    "b" : 1.,
    "C[0]" : 0,
    "C[1]" : 0,
    'D_alpha[0]': 0,
    'D_alpha[1]': 0
  },
  "output" : {
    "filename" : "sim_results/traction_ex2.1_q5.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 200,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        2 : {
            "time" : 200,
            "load" : 0.02,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

nli_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
nli_instance.print_input()
nli_instance.run_simulation()
data_nli = nli_instance.print_output()

In [ ]:
nli_instance.graph_e_and_sigma_time(data_nli)

In [ ]:
nli_instance.graph_e_and_sigma(dataframe = data_nli, 
                               input_data = "data/exercice1_traction.csv", 
                               savefig_name = "sim_results/traction_ex2.1_q5.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.1 · Q5</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `Q` | *à compléter* |
| `b` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.1 Traction · Q6 / Q6</span>

#### Q6 · Écrouissages combinés

<div class="alert alert-block alert-info">

À l'aide de vos premières observations, essayez de trouver le meilleur compromis en **combinant les écrouissages**. Que pouvez-vous en conclure sur l'identification de paramètres à partir d'un essai de traction ?

🧩 **Modèle :** `asternonlinearisotropicnonlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** tous

</div>

<div class="alert alert-block alert-warning">

**💡 Indice :** vous pouvez même utiliser des modèles linéaires !

</div>

In [ ]:
#########################################################################
### Simulation d'un essai de traction - 
### écrouissage cinématique non linéaire combiné avec un écrouissage 
### isotrope non linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 200000,
    "PoissonRatio" : 0.3,
    "R0" : 10000,
    "Q" : 1000,
    "b" : 1.,
    "C[0]" : 1000,
    "C[1]" : 0,
    'D_alpha[0]': 1000,
    'D_alpha[1]': 0
  },
  "output" : {
    "filename" : "sim_results/traction_ex2.1_q6.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 200,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        2 : {
            "time" : 200,
            "load" : 0.02,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

mixed_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
mixed_instance.print_input()
mixed_instance.run_simulation()
data_mixed = mixed_instance.print_output()

In [ ]:
mixed_instance.graph_e_and_sigma_time(data_mixed)

In [ ]:
mixed_instance.graph_e_and_sigma(dataframe = data_mixed, 
                                 input_data = "data/exercice1_traction.csv", 
                                 savefig_name = "sim_results/traction_ex2.1_q6.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.1 · Q6</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `Q` | *à compléter* |
| `b` | *à compléter* |
| `C[0]` | *à compléter* |
| `D_alpha[0]` | *à compléter* |
| `C[1]` | *à compléter* |
| `D_alpha[1]` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

<h3 style="border-left:6px solid #1f4e79;padding-left:10px">🔁 2.2 — Autour d'un chargement cyclique</h3>

In [ ]:
from src.code.functions import draw_data
draw_data(input_data = "data/exercice2_cyclique.csv", figure_title = 'FIGURE 3: Chargement et réponse mécanique étudiés')

<div class="alert alert-block alert-info">

**📌 Chargement :** chargement cyclique **piloté en déformation entre 0 et 0,005 mm/mm (soit 0,5 %)**, à vitesse constante :

| Temps (s) | 0 → 10 | 10 → 20 | 20 → 30 | 30 → 40 | 40 → 50 |
|:--|:--:|:--:|:--:|:--:|:--:|
| Déformation (mm/mm) | 0 → 0,005 | 0,005 → 0 | 0 → 0,005 | 0,005 → 0 | 0 → 0,005 |

</div>

### ❓ Questions

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.2 Cyclique · Q1 / Q6</span>

#### Q1 · Module d'Young

<div class="alert alert-block alert-info">

Estimez tout d'abord la valeur du **module d'Young** $E$ en considérant les premiers points expérimentaux, dans le régime linéaire.

</div>

> 💡 Les essais étant simulés, il n'y a **pas de bruit** : vous devez trouver une valeur **exacte**.

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.2 · Q1</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

**Module d'Young :** $E =$ *à compléter* MPa

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.2 Cyclique · Q2 / Q6</span>

#### Q2 · Écrouissage cinématique linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **cinématique linéaire** : faites varier la limite d'élasticité et le module d'écrouissage. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ?

🧩 **Modèle :** `asterlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `C`

</div>

In [ ]:
#########################################################################
### Simulation d'un essai cyclique - écrouissage cinématique linéaire 
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 150000,
    "PoissonRatio" : 0.3,
    "R0" : 350,
    "C" : 8000
  },
  "output" : {
    "filename" : "sim_results/cyclique_ex2.2_q2.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 50,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        6 : {
            "time" : 50,
            "load" : 0.005,
        },
        5 : {
            "time" : 40,
            "load" : 0.0,
        },
        4 : {
            "time" : 30,
            "load" : 0.005,
        },
        3 : {
            "time" : 20,
            "load" : 0,
        },
        2 : {
            "time" : 10,
            "load" : 0.005,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

clk_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asterlinearkinematic')
clk_instance.print_input()
clk_instance.run_simulation()
data_clk = clk_instance.print_output()

In [ ]:
clk_instance.graph_e_and_sigma_time(data_clk)

In [ ]:
clk_instance.graph_e_and_sigma(dataframe = data_clk, 
                               input_data = "data/exercice2_cyclique.csv", 
                               savefig_name = "sim_results/cyclique_ex2.2_q2.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.2 · Q2</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `C` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.2 Cyclique · Q3 / Q6</span>

#### Q3 · Écrouissage cinématique non linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **cinématique non linéaire** : faites varier la limite d'élasticité, le module d'écrouissage et le terme de rappel. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ?

🧩 **Modèle :** `asternonlinearisotropicnonlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `C[0]`, `D_alpha[0]` (garder `Q = b = 0`)

</div>

In [ ]:
#########################################################################
### Simulation d'un essai cyclique - écrouissage cinématique non linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 20000,
    "PoissonRatio" : 0.3,
    "R0" : 1000,
    "C[0]" : 1000,
    "D_alpha[0]" : 1000.,
    "Q" : 0,
    "b" : 0,
    "C[1]" : 0,
    "D_alpha[1]" : 0
  },
  "output" : {
    "filename" : "sim_results/cyclique_ex2.2_q3.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 50,
    "step_time": 1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        6 : {
            "time" : 50,
            "load" : 0.005,
        },
        5 : {
            "time" : 40,
            "load" : 0.0,
        },
        4 : {
            "time" : 30,
            "load" : 0.005,
        },
        3 : {
            "time" : 20,
            "load" : 0,
        },
        2 : {
            "time" : 10,
            "load" : 0.005,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

cnlk_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
cnlk_instance.print_input()
cnlk_instance.run_simulation()
data_cnlk = cnlk_instance.print_output()

In [ ]:
cnlk_instance.graph_e_and_sigma_time(data_cnlk)

In [ ]:
cnlk_instance.graph_e_and_sigma(dataframe = data_cnlk, 
                                input_data = "data/exercice2_cyclique.csv", 
                                savefig_name = "sim_results/cyclique_ex2.2_q3.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.2 · Q3</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `C[0]` | *à compléter* |
| `D_alpha[0]` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.2 Cyclique · Q4 / Q6</span>

#### Q4 · Écrouissage isotrope linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **isotrope linéaire** : faites varier la limite d'élasticité et la constante de durcissement. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ? Quelles différences obtenez-vous avec un modèle cinématique ?

🧩 **Modèle :** `asterlinearisotropic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `K`

</div>

In [ ]:
#########################################################################
### Simulation d'un essai cyclique - écrouissage isotrope linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 20000,
    "PoissonRatio" : 0.3,
    "R0" : 1000,
    "K" : 1000,
  },
  "output" : {
    "filename" : "sim_results/cyclique_ex2.2_q4.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 50,
    "step_time": 0.5
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        6 : {
            "time" : 50,
            "load" : 0.005,
        },
        5 : {
            "time" : 40,
            "load" : 0.0,
        },
        4 : {
            "time" : 30,
            "load" : 0.005,
        },
        3 : {
            "time" : 20,
            "load" : 0,
        },
        2 : {
            "time" : 10,
            "load" : 0.005,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

cli_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asterlinearisotropic')
cli_instance.print_input()
cli_instance.run_simulation()
data_cli = cli_instance.print_output()

In [ ]:
cli_instance.graph_e_and_sigma_time(data_cli)

In [ ]:
cli_instance.graph_e_and_sigma(dataframe = data_cli, 
                               input_data = "data/exercice2_cyclique.csv", 
                               savefig_name = "sim_results/cyclique_ex2.2_q4.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.2 · Q4</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `K` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.2 Cyclique · Q5 / Q6</span>

#### Q5 · Écrouissage isotrope non linéaire

<div class="alert alert-block alert-info">

Simulez l'essai avec un modèle d'écrouissage **isotrope non linéaire** : faites varier la limite d'élasticité, la constante de durcissement et le terme de rappel. Quel est le meilleur compromis que vous pouvez obtenir avec ce modèle ?

🧩 **Modèle :** `asternonlinearisotropicnonlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** `R0`, `Q`, `b` (garder `C[i] = D_alpha[i] = 0`)

</div>

In [ ]:
#########################################################################
### Simulation d'un essai cyclique - écrouissage isotrope non linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 20000,
    "PoissonRatio" : 0.3,
    "R0" : 1000,
    "Q" : 1000,
    "b" : 1.,
    "C[0]" : 0,
    "C[1]" : 0,
    'D_alpha[0]': 0,
    'D_alpha[1]': 0
  },
  "output" : {
    "filename" : "sim_results/cyclique_ex2.2_q5.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 50,
    "step_time": 0.5
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        6 : {
            "time" : 50,
            "load" : 0.005,
        },
        5 : {
            "time" : 40,
            "load" : 0.0,
        },
        4 : {
            "time" : 30,
            "load" : 0.005,
        },
        3 : {
            "time" : 20,
            "load" : 0,
        },
        2 : {
            "time" : 10,
            "load" : 0.005,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

cnli_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
cnli_instance.print_input()
cnli_instance.run_simulation()
data_cnli = cnli_instance.print_output()

In [ ]:
cnli_instance.graph_e_and_sigma_time(data_cnli)

In [ ]:
cnli_instance.graph_e_and_sigma(dataframe = data_cnli, 
                                input_data = "data/exercice2_cyclique.csv", 
                                savefig_name = "sim_results/cyclique_ex2.2_q5.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.2 · Q5</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `Q` | *à compléter* |
| `b` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<span style="background-color:#1f4e79;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 2 › 2.2 Cyclique · Q6 / Q6</span>

#### Q6 · Écrouissages combinés

<div class="alert alert-block alert-info">

À l'aide de vos premières observations, essayez de trouver le meilleur compromis en **combinant les écrouissages**. Que pouvez-vous en conclure sur l'identification de paramètres à partir d'un essai cyclique ?

🧩 **Modèle :** `asternonlinearisotropicnonlinearkinematic`<br>
🎛️ **Paramètres à ajuster :** tous

</div>

<div class="alert alert-block alert-warning">

**💡 Indice :** vous pouvez même utiliser des modèles linéaires !

</div>

In [ ]:
#########################################################################
### Simulation d'un essai cyclique - écrouissage cinématique non linéaire
### combiné avec un écrouissage isotrope non linéaire
#########################################################################

from src.code.models import lk

model_dict = {
  "material_property" : {
    "YoungModulus" : 100000,
    "PoissonRatio" : 0.3,
    "R0" : 500,
    "Q" : 1000,
    "b" : 1.,
    "C[0]" : 1000,
    "C[1]" : 0,
    'D_alpha[0]': 1000,
    'D_alpha[1]': 0
  },
  "output" : {
    "filename" : "sim_results/cyclique_ex2.2_q6.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 50,
    "step_time": 0.5
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        6 : {
            "time" : 50,
            "load" : 0.005,
        },
        5 : {
            "time" : 40,
            "load" : 0.0,
        },
        4 : {
            "time" : 30,
            "load" : 0.005,
        },
        3 : {
            "time" : 20,
            "load" : 0,
        },
        2 : {
            "time" : 10,
            "load" : 0.005,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

cmixed_instance = lk(model_input_file = model_dict, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
cmixed_instance.print_input()
cmixed_instance.run_simulation()
data_cmixed = cmixed_instance.print_output()

In [ ]:
cmixed_instance.graph_e_and_sigma_time(data_cmixed)

In [ ]:
cmixed_instance.graph_e_and_sigma(dataframe = data_cmixed, 
                                  input_data = "data/exercice2_cyclique.csv", 
                                  savefig_name = "sim_results/cyclique_ex2.2_q6.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — 2.2 · Q6</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `Q` | *à compléter* |
| `b` | *à compléter* |
| `C[0]` | *à compléter* |
| `D_alpha[0]` | *à compléter* |
| `C[1]` | *à compléter* |
| `D_alpha[1]` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

<h2 style="background-color:#5b3a7a;color:#fff;padding:10px;border-radius:6px;text-align:center">🏗️ Exercice 3 — Corrélation calcul/essais en plasticité</h2>

Il s'agit maintenant de mettre en pratique ce que vous avez appris dans l'exercice 2 pour identifier un comportement sur des **données expérimentales réelles** : deux essais réalisés sur des éprouvettes en **fonte**.

<div class="alert alert-block alert-warning">

**⚠️ Consigne :** utilisez **uniquement un modèle de plasticité** et essayez de reproduire au mieux **les deux essais avec un même jeu de paramètres**.

</div>

In [ ]:
from src.code.functions import draw_data
draw_data(input_data = "data/exercice3_traction.csv", figure_title = 'FIGURE 4: Chargement et réponse mécanique pour une fonte en traction')

In [ ]:
from src.code.functions import draw_data
draw_data(input_data = "data/exercice3_cyclique.csv", figure_title = 'FIGURE 5: Chargement et réponse mécanique pour une fonte sous chargement cyclique')

---

<span style="background-color:#5b3a7a;color:#fff;padding:2px 10px;border-radius:12px;font-size:85%">Ex. 3 › Fonte · Q1 – Q2</span>

### ❓ Questions

| | Essai | Chargement | Figure |
|:--:|:--|:--|:--:|
| **Q1** | Traction | pilotage en déformation imparfait → linéarisé de 0 à 0,02 mm/mm en 200 s | 4 |
| **Q2** | Cyclique | traction-compression alternée entre −0,005 et +0,005 mm/mm, sur 100 s | 5 |

1. Il s'agit tout d'abord d'un essai de traction pour lequel le pilotage en déformation est imparfait, comme on peut le constater sur la Figure 4. Le chargement est modélisé en linéarisant l'évolution de la déformation : $\varepsilon(t) = \dot{\varepsilon}_0\, t$, de 0 à 0,02 mm/mm en 200 s.

2. Vous disposez également d'un essai cyclique de **traction-compression entre −0,005 et +0,005 mm/mm**, réalisé à vitesse constante (Figure 5) :

| Temps (s) | 0 → 20 | 20 → 40 | 40 → 60 | 60 → 80 | 80 → 100 |
|:--|:--:|:--:|:--:|:--:|:--:|
| Déformation (mm/mm) | 0 → −0,005 | −0,005 → 0,005 | 0,005 → −0,005 | −0,005 → 0,005 | 0,005 → −0,005 |

<div class="alert alert-block alert-info">

**ℹ️ Paramètres partagés :** les paramètres matériau sont définis **une seule fois**, dans le bloc *Variables globales à changer* en haut de la cellule suivante. Ils sont utilisés pour **les deux** simulations (traction et cyclique).

</div>

In [ ]:
#########################################################################
### Simulation d'un essai de traction pour de la 
### fonte - écrouissage cinématique non linéaire combiné avec un 
### écrouissage isotrope non linéaire
#########################################################################

from src.code.models import lk

#########################################################################
### Variables globales à changer

young_modulus = 200000
poisson_ration = 0.3
R0 = 600
Q = 1000
b = 1
C_0 = 100
C_1 = 0
D_alpha_0 = 1000
D_alpha_1 = 0

#########################################################################

model_dict_traction = {
  "material_property" : {
    "YoungModulus" : young_modulus,
    "PoissonRatio" : poisson_ration,
    "R0" : R0,
    "Q" : Q,
    "b" : b,
    "C[0]" : C_0,
    "C[1]" : C_1,
    'D_alpha[0]': D_alpha_0,
    'D_alpha[1]': D_alpha_1
  },
  "output" : {
    "filename" : "sim_results/traction_ex3_q1.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 200,
    "step_time": 0.1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        2 : {
            "time" : 200,
            "load" : 0.02,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

tmixedi_instance = lk(model_input_file = model_dict_traction, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
tmixedi_instance.print_input()
tmixedi_instance.run_simulation()
data_tmixedi = tmixedi_instance.print_output()

#########################################################################
### Simulation d'un essai cyclique pour de la 
### fonte - écrouissage cinématique non linéaire combiné avec un 
### écrouissage isotrope non linéaire
#########################################################################

model_dict_cyclique = {
  "material_property" : {
    "YoungModulus" : young_modulus,
    "PoissonRatio" : poisson_ration,
    "R0" : R0,
    "Q" : Q,
    "b" : b,
    "C[0]" : C_0,
    "C[1]" : C_1,
    'D_alpha[0]': D_alpha_0,
    'D_alpha[1]': D_alpha_1
  },
  "output" : {
    "filename" : "sim_results/cyclique_ex3_q2.txt"
  },
  "external_state_variable" : {
    "Temperature" : 293.15
  },
  "time" : {
    "start_time" : 0.0,
    "end_time" : 100,
    "step_time": 0.1
  },
  "load" : {
    "method" : "strain",
    "direction" : "EXX",
    "loading_blocks" : {
        6 : {
            "time" : 100,
            "load" : -0.005,
        },
        5 : {
            "time" : 80,
            "load" : 0.005,
        },
        4 : {
            "time" : 60,
            "load" : -0.005,
        },
        3 : {
            "time" : 40,
            "load" : 0.005,
        },
        2 : {
            "time" : 20,
            "load" : -0.005,
        },
        1 : {
            "time" : 0.0,
            "load" : 0.0,
        }
    }
  }
}

cmixedi_instance = lk(model_input_file = model_dict_cyclique, 
                 behavior_name = 'asternonlinearisotropicnonlinearkinematic')
cmixedi_instance.print_input()
cmixedi_instance.run_simulation()
data_cmixedi = cmixedi_instance.print_output()

In [ ]:
tmixedi_instance.graph_e_and_sigma(dataframe = data_tmixedi, 
                                   input_data = "data/exercice3_traction.csv", 
                                   savefig_name = "sim_results/traction_ex3_q1.png")

In [ ]:
cmixedi_instance.graph_e_and_sigma(dataframe = data_cmixedi, 
                                   input_data = "data/exercice3_cyclique.csv", 
                                   savefig_name = "sim_results/cyclique_ex3_q2.png")

<div style="border:2px solid #2e7d32;border-left:10px solid #2e7d32;background-color:#2e7d3214;padding:10px 18px;border-radius:8px">

<span style="background-color:#2e7d32;color:#fff;padding:3px 12px;border-radius:12px;font-weight:bold">✍️ VOTRE RÉPONSE — Exercice 3 · Q1 – Q2</span>

<!-- ✏️ Remplacez les textes « à compléter » / « Écrivez… » par vos réponses (Markdown et formules $...$ acceptés).
     Ne supprimez pas la dernière ligne (fermeture du cadre) et gardez une ligne vide juste avant. -->

| Paramètre retenu | Valeur |
|:--|:--|
| `YoungModulus` | *à compléter* |
| `R0` | *à compléter* |
| `Q` | *à compléter* |
| `b` | *à compléter* |
| `C[0]` | *à compléter* |
| `D_alpha[0]` | *à compléter* |
| `C[1]` | *à compléter* |
| `D_alpha[1]` | *à compléter* |

**Commentaire :**

*Écrivez votre commentaire ici…*

</div>

---

<div class="alert alert-block alert-success">

**✅ Avant de rendre votre compte rendu**

- ☐ Vérifiez que **toutes les cellules ont été exécutées sans erreur**. En cas d'erreur, contactez [baris.telmen@ensta.fr](mailto:baris.telmen@ensta.fr).
- ☐ Vous pouvez **rédiger vos commentaires directement dans les cadres verts** *VOTRE RÉPONSE* de ce notebook, **ou** préparer votre propre compte rendu à partir des données générées dans le dossier `sim_results/`.
- ☐ Les **figures générées** par ce notebook sont enregistrées dans `sim_results/`. Vous pouvez les utiliser telles quelles ou produire vos propres figures, à condition qu'elles soient correctement présentées.

</div>

<div class="alert alert-block alert-warning">

**🌐 Accès aux serveurs**

- Pour accéder aux serveurs, vous devez être connecté au réseau **EDUROAM de l'ENSTA**. Les points d'accès EDUROAM des **autres établissements ne fonctionneront pas**.
- En cas de problème d'accès aux serveurs, contactez [baris.telmen@ensta.fr](mailto:baris.telmen@ensta.fr).

</div>